In [3]:
from pathlib import Path
import sys
from config.settings import get_settings
from src.ingestion.connectors import CelonisConnector

In [4]:
repo_root = Path().resolve()
sys.path.insert(0, str(repo_root))

settings = get_settings()

connector = CelonisConnector(
    base_url=settings.celonis_url,
    api_token=settings.celonis_api_token,
    key_type=getattr(settings, "celonis_key_type", "USER_KEY"),
)
connector.connect()

In [5]:
if settings.space_id:
    space = connector.get_space(settings.space_id)
    print(f"Space: {space.name} (id={settings.space_id})")
else:
    print("Space: SKIPPED (SPACE_ID not set in .env)")

Space: S2P Demo (id=02859430-a8a0-4960-9b2a-04130320181a)


In [6]:
if settings.space_id and settings.package_id:
    package = connector.get_package(settings.space_id, settings.package_id)
    print(f"Package: {package.name} (id={settings.package_id})")
elif not settings.package_id:
    print("Package: SKIPPED (PACKAGE_ID not set in .env)")
else:
    print("Package: SKIPPED (SPACE_ID not set in .env)")

Package: S2P Demo (id=0909cf90-787f-40b5-a020-1a1b80ce05dc)


In [7]:
pool_identifier = settings.data_pool_id or settings.data_pool_name
if pool_identifier:
    pool = connector.get_data_pool(pool_identifier)
    print(f"Data Pool: {pool.name} (identifier={pool_identifier})")
else:
    print("Data Pool: SKIPPED (DATA_POOL_ID or DATA_POOL_NAME not set in .env)")

Data Pool: S2P Demo Data (identifier=c0f753bd-db9d-4a4e-ab63-0a636588a478)


In [8]:
import json

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [9]:
kpi_data = load_json("data/raw/extracted_kpis.json")
kpis_only = [x for x in kpi_data if str(x.get("attribute_type", "")).lower() == "kpi"]

print(f"Total KPIs: {len(kpis_only)}")
kpis_only[:]  

Total KPIs: 16


[{'kpi_id': 'po_spend_',
  'name': 'PO Spend %',
  'pql_formula': 'COUNT("o_custom_AuroraInvoiceKpiAllinone"."ID")/SUM("o_custom_AuroraInvoiceKpiAllinone"."QUANTITY")*20',
  'description': 'None',
  'record_id': 'knowledge_model',
  'attribute_type': 'KPI',
  'raw_metadata': {},
  'extracted_at': '2026-03-26 11:29:44.567692'},
 {'kpi_id': 'po_spend',
  'name': 'PO Spend',
  'pql_formula': 'KPI("condition_yes_po")',
  'description': 'None',
  'record_id': 'knowledge_model',
  'attribute_type': 'KPI',
  'raw_metadata': {},
  'extracted_at': '2026-03-26 11:29:44.567733'},
 {'kpi_id': 'non_po_spend',
  'name': 'Non PO Spend',
  'pql_formula': 'SUM("o_custom_AuroraInvoiceKpiAllinone"."non_po_spend")',
  'description': 'None',
  'record_id': 'knowledge_model',
  'attribute_type': 'KPI',
  'raw_metadata': {},
  'extracted_at': '2026-03-26 11:29:44.567745'},
 {'kpi_id': 'invoice_value',
  'name': 'Invoice Value',
  'pql_formula': 'KPI("po_spend")*5',
  'description': 'None',
  'record_id': 'kn

In [11]:
kpi_data = load_json("data/raw/extracted_kpis.json")
attributes_only = [x for x in kpi_data if str(x.get("attribute_type", "")).lower() == "attribute"]

print(f"Total Attributes: {len(attributes_only)}")
attributes_only[:]

Total Attributes: 50


[{'kpi_id': 'ID',
  'name': 'Id',
  'pql_formula': '"o_custom_AuroraInvoiceKpiAllinone"."ID"',
  'description': 'None',
  'record_id': 'O_CUSTOM_AURORAINVOICEKPIALLINONE',
  'attribute_type': 'Attribute',
  'raw_metadata': {},
  'extracted_at': '2026-03-26 11:29:44.569874'},
 {'kpi_id': 'ITEMID',
  'name': 'Itemid',
  'pql_formula': '"o_custom_AuroraInvoiceKpiAllinone"."ITEMID"',
  'description': 'None',
  'record_id': 'O_CUSTOM_AURORAINVOICEKPIALLINONE',
  'attribute_type': 'Attribute',
  'raw_metadata': {},
  'extracted_at': '2026-03-26 11:29:44.569890'},
 {'kpi_id': 'DOCID',
  'name': 'Docid',
  'pql_formula': '"o_custom_AuroraInvoiceKpiAllinone"."DOCID"',
  'description': 'None',
  'record_id': 'O_CUSTOM_AURORAINVOICEKPIALLINONE',
  'attribute_type': 'Attribute',
  'raw_metadata': {},
  'extracted_at': '2026-03-26 11:29:44.569898'},
 {'kpi_id': 'COMPANYCODE',
  'name': 'Companycode',
  'pql_formula': '"o_custom_AuroraInvoiceKpiAllinone"."COMPANYCODE"',
  'description': 'None',
  'r

In [13]:
transformations = load_json("data/raw/extracted_transformations.json")

print(f"Total Transformations: {len(transformations)}")
transformations[:] 


Total Transformations: 32


[{'job_name': 'Aug Validation',
  'transformation_name': 'Validation',
  'sap_tables_used': 'N/A',
  'status': 'Success',
  'raw_sql': 'SELECT * FROM "AA_HISTORY_LOG"',
  'extracted_at': '2026-03-26 11:29:47.607909'},
 {'job_name': 'test:ocpm-data-job',
  'transformation_name': 'transformation_ENTITY_OBJECT_custom_MasterP2P',
  'sap_tables_used': 'N/A',
  'status': 'Success',
  'raw_sql': '-- 1) create tables\nCREATE TABLE IF NOT EXISTS "c0f753bd-db9d-4a4e-ab63-0a636588a478_OCDM"."t_o_custom_MasterP2P" ("LogTimestamp" TIMESTAMP,\n    "Role" VARCHAR(29),\n    "LogUser" VARCHAR(80),\n    "LogInefficiencyReason" VARCHAR(80),\n    "LogActivity" VARCHAR(80),\n    "AIRecommendation" VARCHAR(113),\n    "AIrootCauseInsight" VARCHAR(93),\n    "CashAtRisk" FLOAT,\n    "PredictedDelay" BIGINT,\n    "ForecastedPaymentDate" TIMESTAMP,\n    "SupplierScore" BIGINT,\n    "OperationalWasteCost" FLOAT,\n    "DiscountLost" FLOAT,\n    "PriceLeakage" FLOAT,\n    "PoCreationDelayReason" VARCHAR(80),\n    "

In [14]:
with open("data/raw/extracted_transformations.json", "r", encoding="utf-8") as f:
    transformations = json.load(f)

job_names = sorted({t.get("job_name") for t in transformations if t.get("job_name")})
print(f"Total unique data jobs: {len(job_names)}")
for name in job_names:
    print(name)

Total unique data jobs: 4
Aug Validation
Predictions
ocpm-data-job
test:ocpm-data-job
